# as-strided-noncontig-source — ex10: writes through overlapping strided view corrupt the source

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-noncontig-source`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## as_strided write-aliasing — quick refresher

`as_strided(x, size, stride)` constructs a view whose elements may share underlying storage with `x`. When the strides cause **overlap** (adjacent output elements map to the same source element), an in-place write through the view will corrupt every overlap-sharing output position — even ones written 'earlier' in your code.

```python
x = torch.arange(6).float()                      # [0,1,2,3,4,5]
w = x.as_strided((4, 3), (1, 1))                  # 4 overlapping windows
w[0, 0] = 99.0                                    # writes x[0] = 99
# w[0] = [99,1,2], but also EVERY window starting at offset 0 sees 99
```

**Compared to non-overlapping strides.** If the strides are at least as large as the per-row size, the view is a non-overlapping partition — in-place writes behave normally. Overlap is what makes `as_strided` a footgun.

**This drill (ex10) vs ex1-9.** Earlier exercises read from as_strided views (sliding windows, convolution, diagonal extraction). ex10 deliberately writes through an overlapping view to demonstrate aliasing, then asks the caller to characterize the corruption pattern.

### Exercise 10 — writes through overlapping strided view corrupt the source

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the corruption produced by writing through an overlapping as_strided view, and return both the post-write source vector and a boolean mask of which source positions were mutated.
> Keywords: aliasing, in-place-write, overlap, footgun
> ```

**KCs targeted:** `as-strided-shares-storage`, `as-strided-overlap-write-aliasing`

Implement `ex10_overlap_write_demo(n, window, target_window, fill_value)`.

Demonstrate the as_strided write-aliasing pitfall:

1. Allocate `x = t.arange(n, dtype=t.float32)`.
2. Build a sliding-window view `w = x.as_strided((n - window + 1, window), (1, 1))` — stride 1 in both dims = full overlap.
3. Save a copy `x_before = x.clone()` so you can diff later.
4. Mutate **only** `w[target_window]` in place: `w[target_window] = fill_value`. This is a row-write that hits the overlapping storage.
5. After the write, return `(x, mutated_mask)` where:
   - `x` is the source vector AFTER the write (now corrupted).
   - `mutated_mask` is a bool tensor of length `n`, `True` at every position where `x[i] != x_before[i]`.

The visualization plots `x_before` and `x` after the write to make the corruption visually obvious — the contiguous band of fill_value is exactly the storage range covered by `w[target_window]`.

In [ ]:
def ex10_overlap_write_demo(
    n: int,
    window: int,
    target_window: int,
    fill_value: float,
) -> tuple[Tensor, Tensor]:
    """Write through one overlapping window; return (corrupted_x, mutated_mask)."""
    raise NotImplementedError()


def _test_ex10():
    # Canonical case: n=10, window=3, target_window=4, fill=-1
    # w[4] aliases x[4], x[5], x[6] — those three positions should mutate.
    x, mask = ex10_overlap_write_demo(10, 3, 4, -1.0)
    assert x.shape == (10,), f'x shape wrong: {tuple(x.shape)}'
    assert mask.shape == (10,), f'mask shape wrong: {tuple(mask.shape)}'
    assert mask.dtype == t.bool, f'mask dtype wrong: {mask.dtype}'

    # Exactly indices 4, 5, 6 should be -1.
    assert x[4].item() == -1.0
    assert x[5].item() == -1.0
    assert x[6].item() == -1.0
    # Unmutated positions should equal arange.
    for i in (0, 1, 2, 3, 7, 8, 9):
        assert x[i].item() == float(i), f'x[{i}] should be {float(i)}, got {x[i].item()}'
    # Mask matches.
    expected_mask = t.tensor([False]*4 + [True]*3 + [False]*3)
    assert t.equal(mask, expected_mask), f'mask mismatch:\n{mask}\nvs\n{expected_mask}'

    # Edge case: target_window=0 → mutates x[0..window-1].
    x2, m2 = ex10_overlap_write_demo(8, 4, 0, 99.0)
    for i in range(4):
        assert x2[i].item() == 99.0
    for i in range(4, 8):
        assert x2[i].item() == float(i)

    # Edge case: target_window at end.
    x3, m3 = ex10_overlap_write_demo(12, 5, 7, 0.0)  # mutates x[7..11]
    assert m3[:7].sum().item() == 0
    assert m3[7:].sum().item() == 5

    # --- Visualization: before vs after the overlapping write ---
    n, win, tgt, fill = 30, 5, 12, -3.0
    before = t.arange(n, dtype=t.float32)
    after, _ = ex10_overlap_write_demo(n, win, tgt, fill)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(n), before.numpy(), label='x before', color='steelblue', linewidth=2)
    ax.plot(range(n), after.numpy(), label='x after w[12] = -3', color='crimson',
            linewidth=2, linestyle='--')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('source index')
    ax.set_ylabel('value')
    ax.set_title(f'ex10 overlap write: w[{tgt}] = {fill} corrupted indices {tgt}..{tgt+win-1}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex10')
    print("ex10 ✓")

_test_ex10()

<details><summary>Solution</summary>

```python
def ex10_overlap_write_demo(
    n: int,
    window: int,
    target_window: int,
    fill_value: float,
) -> tuple[Tensor, Tensor]:
    x = t.arange(n, dtype=t.float32)
    n_windows = n - window + 1
    w = x.as_strided((n_windows, window), (1, 1))
    x_before = x.clone()
    w[target_window] = fill_value  # writes into x storage at offsets target_window..target_window+window-1
    mutated_mask = x != x_before
    return x, mutated_mask
```

**Why writes through overlapping views corrupt the source.** `as_strided` does NOT copy storage — `w` and `x` share the same underlying buffer. Writing `w[target_window]` mutates `window` consecutive elements of `x`'s storage starting at offset `target_window` (because the stride is 1). Every OTHER window that overlaps that range will also see the new values on its next read.

**This is why you should treat `as_strided` views as read-only.** The PyTorch docs warn about exactly this. If you need an overlapping window for compute (e.g. sliding-window convolution), the safe pattern is: build the view → call `.contiguous()` (or `.clone()`) to materialize a non-aliased copy → mutate that copy.

**Debugging tip.** If a strided computation produces wrong outputs AFTER an in-place op (`*=`, `+=`, scatter, etc.), the first hypothesis should be: 'did I just write through an aliased view?' The mutated_mask returned here is the diagnostic — it tells you exactly which source positions got hit by the view write.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()